# Figure S2: Analog generation parameter sweeps

- **A**: Property deviation (MAE) across tau-sigma
- **B**: Active prototypes, best analog (MIC + Similarity heatmaps)
- **C**: Inactive prototypes, best analog
- **D**: Active prototypes, all analogs (mean +/- std)
- **E**: Inactive prototypes, all analogs

In [ ]:
import pandas as pd, numpy as np, re, warnings
from pathlib import Path
from Bio import SeqIO
from scipy.stats import binned_statistic_2d, gaussian_kde
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import modlamp.analysis as manalysis
warnings.filterwarnings('ignore')

plt.style.use('default')
plt.rcParams.update({
    'font.size':7,'axes.titlesize':7,'axes.labelsize':6,
    'xtick.labelsize':6,'ytick.labelsize':6,'legend.fontsize':6,
    'axes.linewidth':0.5,'axes.edgecolor':'grey',
    'xtick.color':'#555555','ytick.color':'#555555',
    'axes.labelcolor':'#333333','font.family':'sans-serif',
    'font.sans-serif':['Arial','DejaVu Sans'],
    'figure.dpi':300,'savefig.dpi':300,
    'xtick.major.width':0.5,'ytick.major.width':0.5,
    'xtick.major.size':2,'ytick.major.size':2,
    'axes.spines.top':True,'axes.spines.right':True,
    'figure.facecolor':'white','axes.facecolor':'white',
})

ROOT=Path("../data")
SWEEP=ROOT/"figure3/sweep"
CHARGE_GRID=[0,2,4,6,8,10,12]; HYDRO_GRID=[-0.75,-0.50,-0.25,0.0,0.25,0.50,0.75]; LENGTH_GRID=[5,10,15,20,25,30,35]

AGGRESCAN={'I':1.822,'F':1.754,'V':1.594,'L':1.38,'Y':1.159,'W':1.037,'M':0.91,'C':0.604,
           'A':-0.036,'T':-0.159,'S':-0.294,'P':-0.334,'G':-0.535,'K':-0.931,'H':-1.033,
           'Q':-1.231,'R':-1.24,'N':-1.302,'E':-1.412,'D':-1.836}
AA_MW={'A':89.09,'R':174.20,'N':132.12,'D':133.10,'C':121.16,'E':147.13,'Q':146.15,
       'G':75.03,'H':155.16,'I':131.17,'L':131.17,'K':146.19,'M':149.21,'F':165.19,
       'P':115.13,'S':105.09,'T':119.12,'W':204.23,'Y':181.19,'V':117.15}
def peptide_mw(seq): return sum(AA_MW.get(aa,0) for aa in str(seq).upper())-(len(str(seq))-1)*18.015
def bcharge(seqs): h=manalysis.GlobalAnalysis(list(seqs)); h.calc_charge(); return h.charge[0]
def bhydro(seqs): h=manalysis.GlobalAnalysis(list(seqs)); h.calc_H(scale='eisenberg'); return h.H[0]
def seq_agg(s):
    sc=[AGGRESCAN.get(aa,0) for aa in str(s).upper() if aa in AGGRESCAN]
    if len(sc)<5: return np.mean(sc) if sc else np.nan
    return np.mean([np.mean(sc[i:i+5]) for i in range(len(sc)-4)])
def calc_props(seqs):
    s=list(seqs); return pd.DataFrame({'charge':bcharge(s),'length':[len(x) for x in s],'hydrophobicity':bhydro(s)})
def style_spines(ax):
    for sp in ax.spines.values(): sp.set_edgecolor('grey'); sp.set_linewidth(0.5)
def parse_hydro(s):
    if s.startswith('hm'): return -float(s[2:].replace('p','.'))
    elif s.startswith('h'): return float(s[1:].replace('p','.'))
    return None
def parse_dir(dn,pt):
    parts=dn.split('_')
    if pt=='ch': return int(parts[0][1:]),parse_hydro(parts[1])
    elif pt=='cl': return int(parts[0][1:]),int(parts[1][1:])
    elif pt=='hl': return parse_hydro(parts[0]),int(parts[1][1:])
    return None,None
def is_allowed(xt,yt,pt):
    tol=0.01
    if pt=='ch': return xt in CHARGE_GRID and any(abs(yt-h)<tol for h in HYDRO_GRID)
    elif pt=='cl': return xt in CHARGE_GRID and yt in LENGTH_GRID
    elif pt=='hl': return any(abs(xt-h)<tol for h in HYDRO_GRID) and yt in LENGTH_GRID
    return False
PL=dict(fontsize=10,fontweight='bold',va='top',ha='left')


In [ ]:
# ================================================================
# ANALOG DATA
# ================================================================
TAU_VALUES=[0,50,100,150,200,300,500,700]; SIGMA_VALUES=[0.0,0.25,0.5,0.75,1.0]
TAU_PLOT=[0,100,200,300,500,700]
TAU_COLORS={0:'#440154',100:'#365c8d',200:'#1fa187',300:'#4ac16d',500:'#9fda3a',700:'#fde725'}

def compute_grid(pt,best=True):
    rows=[]
    for tau in TAU_VALUES:
        for sigma in SIGMA_VALUES:
            pf=SWEEP/pt/"predictions"/f"{pt}_tau{tau}_sigma{sigma}-min.tsv"
            ff=SWEEP/pt/"sequences"/f"{pt}_tau{tau}_sigma{sigma}.fasta"
            if not pf.exists() or not ff.exists(): continue
            pm={}
            for r in SeqIO.parse(ff,'fasta'):
                parts=r.id.split('|')
                if len(parts)<2: continue
                m=re.search(r'prototype_(\d+)',r.id); pid=int(m.group(1)) if m else -1
                pm[r.id]=(str(r.seq),''.join(c for c in parts[1] if c in 'ACDEFGHIKLMNPQRSTVWY'),pid)
            pdf=pd.read_csv(pf,sep='\t')
            pdf['proto_id']=pdf['Sequence_id'].apply(lambda x:int(re.search(r'prototype_(\d+)',x).group(1)) if re.search(r'prototype_(\d+)',x) else -1)
            sims=[]
            for _,row in pdf.iterrows():
                sid=row['Sequence_id']
                if sid in pm:
                    a,p,_=pm[sid]; ml=max(len(a),len(p))
                    sims.append(sum(1 for x,y in zip(a,p) if x==y)/ml*100 if ml>0 else 0)
                else: sims.append(np.nan)
            pdf['sim']=sims
            if best:
                b=pdf.loc[pdf.groupby('proto_id')['MIC'].idxmin()]
                rows.append({'tau':tau/1000,'sigma':sigma,'mic':np.median(b['MIC']),'sim':np.median(b['sim'].dropna())})
            else:
                g=pdf.groupby('proto_id').agg(mic_mean=('MIC','mean'),mic_std=('MIC','std'),
                                               sim_mean=('sim','mean'),sim_std=('sim','std')).reset_index()
                rows.append({'tau':tau/1000,'sigma':sigma,'mic':g['mic_mean'].median(),'mic_std':g['mic_std'].median(),
                             'sim':g['sim_mean'].median(),'sim_std':g['sim_std'].median()})
    return pd.DataFrame(rows)

def compute_mae():
    rows=[]
    for pt in ['positive','negative']:
        for tau in TAU_VALUES:
            for sigma in SIGMA_VALUES:
                ff=SWEEP/pt/"sequences"/f"{pt}_tau{tau}_sigma{sigma}.fasta"
                if not ff.exists(): continue
                recs=list(SeqIO.parse(ff,'fasta')); an=[]; pr=[]
                for r in recs:
                    parts=r.id.split('|')
                    if len(parts)<2: continue
                    an.append(str(r.seq)); pr.append(''.join(c for c in parts[1] if c in 'ACDEFGHIKLMNPQRSTVWY'))
                if not an: continue
                ac2=bcharge(an); pc2=bcharge(pr); ah2=bhydro(an); ph2=bhydro(pr)
                al2=np.array([len(s) for s in an]); pl2=np.array([len(s) for s in pr])
                for i in range(len(an)):
                    rows.append({'tau':tau,'sigma':sigma,'dc':abs(ac2[i]-pc2[i]),'dl':abs(al2[i]-pl2[i]),'dh':abs(ah2[i]-ph2[i])})
    return pd.DataFrame(rows)

print("\n=== Analog data ===")
gp_b=compute_grid('positive',True); gn_b=compute_grid('negative',True)
gp_a=compute_grid('positive',False); gn_a=compute_grid('negative',False)
mae_df=compute_mae()
mae_agg=mae_df.groupby(['tau','sigma']).agg(
    dc_mean=('dc','mean'),dc_std=('dc','std'),dl_mean=('dl','mean'),dl_std=('dl','std'),
    dh_mean=('dh','mean'),dh_std=('dh','std')).reset_index()
print(f"  Best: {len(gp_b)}/{len(gn_b)}, All: {len(gp_a)}/{len(gn_a)}, MAE: {len(mae_df)}")

In [ ]:
def plot_hm(ax,df,metric,cmap,vmin,vmax,norm=None,fmt='.1f',show_std=False,std_col=None):
    tvs=sorted(df['tau'].unique()); svs=sorted(df['sigma'].unique(),reverse=True)
    mat=np.full((len(svs),len(tvs)),np.nan); smat=np.full_like(mat,np.nan)
    for _,row in df.iterrows():
        ti=tvs.index(row['tau']); si=svs.index(row['sigma'])
        mat[si,ti]=row[metric]
        if show_std and std_col and std_col in row.index: smat[si,ti]=row[std_col]
    kw=dict(norm=norm) if norm else dict(vmin=vmin,vmax=vmax)
    im=ax.imshow(mat,aspect='auto',cmap=cmap,interpolation='nearest',**kw)
    for si in range(len(svs)):
        for ti in range(len(tvs)):
            val=mat[si,ti]
            if np.isnan(val): continue
            nv=np.clip(norm(val),0,1) if norm else np.clip((val-vmin)/(vmax-vmin),0,1)
            rgba=plt.get_cmap(cmap)(nv)
            lum=0.299*rgba[0]+0.587*rgba[1]+0.114*rgba[2]
            tc='white' if lum<0.5 else 'black'
            if show_std and not np.isnan(smat[si,ti]):
                ax.text(ti,si,f"{val:{fmt}}\n\u00b1{smat[si,ti]:{fmt}}",ha='center',va='center',fontsize=6,color=tc,linespacing=0.85)
            else:
                ax.text(ti,si,f"{val:{fmt}}",ha='center',va='center',fontsize=6,color=tc)
    ax.set_xticks(range(len(tvs))); ax.set_xticklabels([f"{t}" for t in tvs],fontsize=6)
    ax.set_yticks(range(len(svs))); ax.set_yticklabels([f"{s}" for s in svs],fontsize=6)
    ax.set_xlabel('\u03C4',fontsize=6); ax.set_ylabel('\u03C3',fontsize=6)
    style_spines(ax)
    return im

In [ ]:
# ================================================================
# FIGURE S2: ANALOG  6.5 x 8
# Use separate GridSpecs with explicit coordinates -- no overlap
# ================================================================
print("\n=== Figure S2 ===")
fig=plt.figure(figsize=(6.5,8.0),dpi=300)

# Vertical budget (in figure fraction, total = 1.0 = 8 inches):
# top margin: 0.02
# A (MAE): 0.10
# legend: 0.03
# gap: 0.02
# B: 0.17
# gap: 0.025
# C: 0.17
# gap: 0.025
# D: 0.17
# gap: 0.025
# E: 0.17
# bottom margin: 0.02
# Total: 0.02+0.10+0.03+0.02+4*0.17+3*0.025+0.02 = 0.02+0.10+0.03+0.02+0.68+0.075+0.02 = 0.945

A_top=0.98;  A_bot=0.865
L_y=0.81  # legend center
B_top=0.775; B_bot=0.635
C_top=0.585; C_bot=0.44
D_top=0.39; D_bot=0.245
E_top=0.195; E_bot=0.05

HM_L=0.08; HM_R=0.91

gs_a_manual = None  # we'll place MAE manually

# MAE panels: 3 axes, centered, spanning the full width
mae_pw=0.23; mae_gap=0.06
mae_total=3*mae_pw+2*mae_gap
mae_x0=HM_L+(HM_R-HM_L-mae_total)/2

fig.text(0.01, A_top, 'A', **PL)
ax_mae=[]
for pi in range(3):
    left_i=mae_x0+pi*(mae_pw+mae_gap)
    ax=fig.add_axes([left_i, A_bot, mae_pw, A_top-A_bot])
    ax_mae.append(ax)

for pi,(prop,ylabel) in enumerate([('dc','\u0394C'),('dl','\u0394L'),('dh','\u0394H')]):
    ax=ax_mae[pi]
    for tau in TAU_PLOT:
        td=mae_agg[mae_agg['tau']==tau].sort_values('sigma')
        if len(td)==0: continue
        ax.errorbar(td['sigma'],td[f'{prop}_mean'],yerr=td[f'{prop}_std'],fmt='o-',
                    color=TAU_COLORS[tau],ms=2.5,lw=0.8,elinewidth=0.4,capsize=1.2,capthick=0.4)
    ax.set_xlabel('\u03C3',fontsize=6); ax.set_ylabel(ylabel,fontsize=7)
    ax.tick_params(labelsize=6); style_spines(ax)

# Legend
tau_handles=[Line2D([],[],marker='o',color=TAU_COLORS[t],lw=0.8,ms=2.5,
             label=f'\u03C4={t/1000:.1f}' if t>0 else '\u03C4=0') for t in TAU_PLOT]
fig.legend(handles=tau_handles,loc='center',bbox_to_anchor=(0.5,L_y),
           ncol=6,fontsize=6,frameon=True,edgecolor='#ccc',fancybox=True,
           handletextpad=0.3,columnspacing=0.8,handlelength=1.5)

# Heatmap rows
mic_norm=TwoSlopeNorm(vcenter=32,vmin=0,vmax=60)

hm_cfgs=[
    (B_top,B_bot,'B',gp_b,'Active (best analog)',False),
    (C_top,C_bot,'C',gn_b,'Inactive (best analog)',False),
    (D_top,D_bot,'D',gp_a,'Active (all analogs)',True),
    (E_top,E_bot,'E',gn_a,'Inactive (all analogs)',True),
]

for top,bot,plabel,df,title,show_std in hm_cfgs:
    h=top-bot
    # Two axes side by side with gap
    ax_w=(HM_R-HM_L-0.12)/2  # each axis width, with gap
    cb_extra=0.03  # room for colorbar

    # MIC (left)
    ax=fig.add_axes([HM_L, bot, ax_w, h])
    fig.text(0.01, top, plabel, **PL)
    im=plot_hm(ax,df,'mic','RdBu',0,60,norm=mic_norm,fmt='.1f',
               show_std=show_std,std_col='mic_std' if show_std else None)
    ax.set_title(title,fontsize=6,pad=3)
    d=make_axes_locatable(ax); cx=d.append_axes('right',size='5%',pad=0.04)
    cb=plt.colorbar(im,cax=cx); cb.set_label('MIC (\u03BCM)',fontsize=6); cb.ax.tick_params(labelsize=6)

    # Similarity (right)
    ax2_left=HM_L+ax_w+0.12
    ax=fig.add_axes([ax2_left, bot, ax_w, h])
    im=plot_hm(ax,df,'sim','YlGn',0,100,fmt='.0f',
               show_std=show_std,std_col='sim_std' if show_std else None)
    ax.set_title(title,fontsize=6,pad=3)
    d=make_axes_locatable(ax); cx=d.append_axes('right',size='5%',pad=0.04)
    cb=plt.colorbar(im,cax=cx); cb.set_label('Similarity (%)',fontsize=6); cb.ax.tick_params(labelsize=6)

out2='../figures/figureS2_analog.png'
plt.savefig(out2,bbox_inches='tight',dpi=300)
plt.savefig(out2.replace('.png','.pdf'),bbox_inches='tight')
print(f"Saved {out2}")
plt.close()
print("\nDone!")